# Coffee Coop — 02. Generación de datos sintéticos

Este notebook genera los scripts `INSERT` para poblar `coffee_coop_db`. Con la configuración por defecto se generan **~5,1 millones de filas**:

| Tabla | Filas aprox. | Cómo se genera |
|---|---:|---|
| `geo.departments` / `geo.municipalities` | 33 / 1.102 | DIVIPOLA completo |
| `farm.coffee_varieties` / `farm.certifications` | 12 / 7 | Catálogos del Source |
| `buy.collection_points` | 180 | Municipios cafeteros |
| `buy.daily_prices` | 312.000 | Un precio por punto y por día (2022-01-01 a 2026-09-30) |
| `farm.farmers` | 150.000 | Nombres y apellidos del Source |
| `farm.farms` | 232.000 | 1 a 4 fincas por caficultor |
| `farm.plots` | 650.000 | Según el área de la finca |
| `farm.plot_varieties` | 1.100.000 | 1 a 3 variedades por lote, porcentajes que suman 100 |
| `farm.farm_certifications` | 110.000 | 22 % de las fincas, con renovaciones |
| `buy.deliveries` | 650.000 | Con estacionalidad de cosecha |
| `buy.quality_analyses` | 649.000 | Una por entrega |
| `pay.settlements` | 588.000 | Una por entrega aceptada |
| `pay.payments` | 692.000 | 1 a 3 pagos por liquidación |

Los datos respetan todas las restricciones del DDL (PK, FK, UNIQUE, CHECK) y también las reglas de negocio que quedan para la unidad de triggers: la liquidación usa el precio del día del mismo punto y fecha, los pagos suman el valor neto, los lotes no superan el área de la finca, etc.

Las funciones están en `scripts/pipelines/01-generate-data/coffee_data_generator.py`. Este notebook las ejecuta paso a paso para poder revisar cada tabla. Si solo quieres generar todo de una vez, usa la última sección o ejecuta el script desde la terminal.

## 1. Rutas e importaciones

En **Colab**: sube la carpeta `coffee-coop` a tu Drive, descomenta las dos líneas de `drive.mount` y cambia `MAIN_PATH`.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import os
import sys
import numpy as np
import pandas as pd

MAIN_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))      # content/coffee-coop
# MAIN_PATH = '/content/drive/MyDrive/sql-101/coffee-coop'              # <- Colab

SOURCE_PATH = os.path.join(MAIN_PATH, 'data', 'source')
SQL_PATH = os.path.join(MAIN_PATH, 'data', 'sql')
GENERATOR_PATH = os.path.join(MAIN_PATH, 'scripts', 'pipelines', '01-generate-data')
sys.path.insert(0, GENERATOR_PATH)

import coffee_data_generator as cdg
print(MAIN_PATH)
print(os.listdir(SOURCE_PATH))

## 2. Configuración

- `scale`: multiplica todos los volúmenes. `1.0` ≈ 5,1 millones de filas; `0.1` ≈ 515.000 (útil para probar).
- `farmers`, `deliveries`, `collection_points`: volúmenes base. El resto (fincas, lotes, pagos…) sale de las probabilidades.
- `seed`: con la misma semilla siempre se generan los mismos datos.
- `rows_per_statement` / `rows_per_file`: filas por cada `INSERT` y por cada archivo `.sql`.

In [ ]:
cfg = cdg.Config(
    seed=42,
    scale=1.0,
    farmers=150_000,
    collection_points=180,
    deliveries=650_000,
    start_date='2022-01-01',
    end_date='2026-09-30',
    rows_per_statement=1_000,
    rows_per_file=250_000,
)
rng = np.random.default_rng(cfg.seed)
cfg

## 3. Cargar el Source

In [ ]:
src = cdg.load_source(SOURCE_PATH)
munis = cdg.coffee_municipalities(src)        # municipios de los departamentos cafeteros, con su peso
veredas = cdg.veredas_by_municipality(src)    # veredas reales por municipio (addresses.xlsx)
print(f'{len(munis)} municipios cafeteros, {len(veredas)} con veredas')
munis.groupby('department_code')['weight'].sum().sort_values(ascending=False).head()

## 4. Función para guardar

Igual que `generate_sql_statement` + `save_sql_to_file` del notebook de Smart Health, pero vectorizada: escribe `INSERT` de 1.000 filas, parte las tablas grandes en archivos de 250.000 filas y envuelve cada archivo en `BEGIN; ... COMMIT;`.

In [ ]:
os.makedirs(SQL_PATH, exist_ok=True)
for f in os.listdir(SQL_PATH):          # limpiar una generación anterior
    if f.endswith('.sql'):
        os.remove(os.path.join(SQL_PATH, f))

TABLES = {key: (order, table, label, cols) for order, table, label, key, cols in cdg.TABLES}

def save_table(key: str, df: pd.DataFrame):
    order, table, label, cols = TABLES[key]
    files = cdg.write_inserts(df, order, table, label, cols, SQL_PATH, cfg.rows_per_statement, cfg.rows_per_file)
    size = sum(os.path.getsize(f) for f in files) / 1024 ** 2
    print(f'✅ {table}: {len(df):,} filas -> {len(files)} archivo(s), {size:.1f} MB')

## 5. Catálogos
### 5.1 Departamentos y municipios

In [ ]:
df_departments = cdg.gen_departments(src)
df_municipalities = cdg.gen_municipalities(src)
save_table('departments', df_departments)
save_table('municipalities', df_municipalities)
df_municipalities.head()

### 5.2 Variedades y certificaciones

In [ ]:
df_varieties = cdg.gen_coffee_varieties(src)
df_certifications = cdg.gen_certifications(src)
save_table('coffee_varieties', df_varieties)
save_table('certifications', df_certifications)
df_certifications

## 6. Compras: puntos de compra y precio del día
### 6.1 Puntos de compra
Al menos uno por departamento cafetero; el resto según el peso del departamento.

In [ ]:
df_points = cdg.gen_collection_points(cfg, src, rng, munis)
save_table('collection_points', df_points)
df_points.head()

### 6.2 Precio del día
Un precio por punto y por día. El precio sigue una tendencia sintética (más bajo en 2023, alza fuerte en 2024-2025) con variación diaria y una pequeña diferencia entre puntos.

In [ ]:
df_prices = cdg.gen_daily_prices(cfg, rng, df_points)
save_table('daily_prices', df_prices)
(df_prices.assign(month=df_prices['price_date'].str[:7])
          .groupby('month')['price_per_kg'].mean().round(0).iloc[::6])

## 7. Producción
### 7.1 Caficultores
Documento único, correo único (70 % tienen), mayores de 18 años en la fecha de ingreso.

In [ ]:
df_farmers = cdg.gen_farmers(cfg, src, rng, munis)
save_table('farmers', df_farmers)
df_farmers.drop(columns='_membership').head()

In [ ]:
df_farmers['gender'].value_counts(normalize=True).round(3)

### 7.2 Fincas
1 a 4 fincas por caficultor, sin repetir el nombre para el mismo dueño. El 75 % está en el municipio donde vive el caficultor. Las veredas salen de `addresses.xlsx` y las coordenadas se ubican alrededor de la zona cafetera del departamento.

In [ ]:
df_farms = cdg.gen_farms(cfg, src, rng, df_farmers, munis, veredas)
save_table('farms', df_farms)
df_farms.head()

In [ ]:
df_farms['total_area_ha'].describe().round(2)

### 7.3 Lotes
El número de lotes depende del área de la finca, y la suma de sus áreas nunca supera el área total.

In [ ]:
df_plots = cdg.gen_plots(cfg, rng, df_farms)
save_table('plots', df_plots)
df_plots.head()

### 7.4 Variedades por lote (N:M)
1 a 3 variedades distintas por lote; los porcentajes de cada lote suman exactamente 100.

In [ ]:
df_plot_varieties = cdg.gen_plot_varieties(cfg, rng, df_plots, df_varieties)
save_table('plot_varieties', df_plot_varieties)
(df_plot_varieties.merge(df_varieties[['variety_id', 'variety_name']], on='variety_id')['variety_name']
                  .value_counts().head(6))

### 7.5 Certificaciones por finca (N:M con vigencia)
Periodos de 3 años con renovaciones; `end_date` queda en `NULL` cuando la certificación sigue vigente.

In [ ]:
df_farm_certs = cdg.gen_farm_certifications(cfg, rng, df_farms, df_certifications)
save_table('farm_certifications', df_farm_certs)
df_farm_certs.drop(columns=['_start', '_end']).head()

## 8. Entregas, calidad y pagos
### 8.1 Entregas
Más entregas en las cosechas (abril-junio y octubre-diciembre). Cada finca entrega casi siempre en el mismo punto de compra. El código sigue el formato `DLV-YYYYMMDD-NNNNNN`.

El `status` se completa en 8.4, después de saber qué entregas se analizaron y liquidaron, por eso las entregas se guardan al final.

In [ ]:
df_deliveries = cdg.gen_deliveries(cfg, rng, df_farms, df_points, munis)
df_deliveries[['delivery_id', 'farm_id', 'collection_point_id', 'delivered_at', 'coffee_type', 'bag_count', 'gross_weight_kg']].head()

### 8.2 Análisis de calidad (1:1)
Las entregas aceptadas tienen humedad entre 10 % y 12,5 %; las rechazadas, humedad fuera de rango o muchos defectos.

In [ ]:
df_analyses = cdg.gen_quality_analyses(cfg, src, rng, df_deliveries)
df_analyses['is_accepted'].value_counts(normalize=True).round(3)

### 8.3 Liquidaciones y pagos
- La liquidación usa el precio del día del mismo punto y fecha, ajustado por el factor de rendimiento y la prima de certificación vigente.
- `paid`: los pagos suman exactamente el valor neto. `partially_paid`: suman menos. `pending` y `cancelled`: sin pagos.

In [ ]:
df_settlements = cdg.gen_settlements(cfg, rng, df_deliveries, df_analyses, df_prices, df_points,
                                     df_farm_certs, df_certifications)
df_payments = cdg.gen_payments(cfg, rng, df_settlements)
df_settlements['status'].value_counts()

### 8.4 Estado de la entrega y guardado
`received` (sin análisis), `rejected`, `accepted` (sin liquidar) o `settled`.

In [ ]:
cdg.finalize_delivery_status(df_deliveries, df_analyses, df_settlements)
print(df_deliveries['status'].value_counts())
save_table('deliveries', df_deliveries)
save_table('quality_analyses', df_analyses)
save_table('settlements', df_settlements)
save_table('payments', df_payments)

## 9. Secuencias y resumen
Las tablas con `SERIAL` se insertan con su id explícito para conocer las llaves foráneas. `99-RESET-SEQUENCES.sql` mueve cada secuencia al último id, para que los `INSERT` que hagan después los estudiantes no choquen.

In [ ]:
cdg.write_reset_sequences(SQL_PATH)
files = sorted(f for f in os.listdir(SQL_PATH) if f.endswith('.sql'))
total_mb = sum(os.path.getsize(os.path.join(SQL_PATH, f)) for f in files) / 1024 ** 2
total_rows = sum(len(x) for x in [df_departments, df_municipalities, df_varieties, df_certifications, df_points,
                                  df_prices, df_farmers, df_farms, df_plots, df_plot_varieties, df_farm_certs,
                                  df_deliveries, df_analyses, df_settlements, df_payments])
print(f'{len(files)} archivos, {total_rows:,} filas, {total_mb:.1f} MB en {SQL_PATH}')
files

## 10. (Opcional) Todo en una sola celda
Equivale a todas las secciones anteriores. Con la misma semilla produce exactamente los mismos datos.

In [ ]:
# data = cdg.generate_all(cfg, SOURCE_PATH)
# summary = cdg.write_all(cfg, data, SQL_PATH)
# summary

## 11. Siguiente paso: cargar en PostgreSQL

Con la base creada (`scripts/ddl/01`, `02` y `03`), ejecuta el pipeline de carga desde `scripts/pipelines/02-insert-data`:

```sh
python sql_insert_pipeline_auto.py --user cc_admin --password "<tu-contraseña>" --db-name coffee_coop_db
```